# Causal Discovery Analysis

## 1. Setup

### 1.1 Prerequisites

Install these first:

- **Install Miniconda** — [docs.conda.io/en/latest/miniconda.html](https://docs.conda.io/en/latest/miniconda.html)
- **Install JDK 21+** (Amazon Corretto 21+ recommended) — [github.com/cmu-phil/tetrad/wiki/Setting-up-Java-for-Tetrad](https://github.com/cmu-phil/tetrad/wiki/Setting-up-Java-for-Tetrad)
- **Verify the Tetrad JAR** — confirm that `notebook/resources/tetrad-current.jar` is the latest version. The current latest jar file can be downloaded from [https://github.com/cmu-phil/py-tetrad/tree/main/pytetrad/resources](https://github.com/cmu-phil/py-tetrad/tree/main/pytetrad/resources); replace the file in `notebook/resources/` if it is out of date.

Make sure `JAVA_HOME` points at the JDK 21 install and that `java -version` prints `21.x`.

### 1.2. Create the environment

From the repo root:

```bash
conda env create -f env.yml
conda activate pykumu
pip install -e .
```

### 1.3. Resources

Make sure the required files are in `notebook/resources/`:

| File | Purpose |
|------|---------|
| `tetrad-current.jar` | Tetrad library |

### 1.4. Run the notebook

Pick whichever matches how you work:

**VS Code** (requires the `ms-python.python` and `ms-toolsai.jupyter` extensions)

1. Open `notebook/issue_causal_analysis.ipynb` in VS Code.
2. Click the kernel picker (top-right of the notebook) and select the **`pykumu`** conda environment.
3. Click **Run All**.


**JupyterLab / Jupyter Notebook in a browser**

From the repo root:

```bash
cd notebook
jupyter lab issue_causal_analysis.ipynb
```

In the browser: **Kernel → Change Kernel → `pykumu`**, then **Run → Run All Cells**.


**Headless (command line, no UI)**

From the repo root:

```bash
cd notebook
jupyter nbconvert --to notebook --execute issue_causal_analysis.ipynb --inplace
```

> ⚠️ In all three cases you must be in the `notebook/` directory (or have it as the working directory), otherwise the notebook can't find `resources/` or write to `pykumu_outputs/`.

> ⚠️ If the causal search runs fail or hang, make sure the `num_threads` parameter on `algorithm.algorithm_boss` / `algorithm.algorithm_fges` is set to `5`.

### 1.5. Verify Outputs

- `pykumu_outputs/` — intermediate CSVs and search graphs (`null_search_graph.json`, `domain_search_graph.json`).
- `causal_graph/` — interactive HTMLs (`causal_graph_full.html`, `causal_graph_subgraph.html`).

### 1.6. Output Comparison with Kumu

To check pykumu matches kumu, run the [Kumu R notebook](https://github.com/phuong808/kumu/blob/R-notebook-display/vignettes/issue_causal_analysis.Rmd) on `1_openssl_social_smells_timeline.csv`. It saves its intermediate CSVs to `kumu_outputs/` next to the `.Rmd` — copy that folder into pykumu's notebook directory:

```bash
cp -r /path/to/kumu/vignettes/kumu_outputs notebook/kumu_outputs
```

After both `notebook/kumu_outputs/` (from R) and `notebook/pykumu_outputs/` (from step 4) exist, run the comparison from the repo root:

```bash
python3 notebook/validation/compare_outputs.py
```

Reports PASS/FAIL per section.


## 2. Introduction

In [154]:
# Start the JVM and load tetrad-current.jar, then import all required packages.
import base64
import os
import sys
import random
import shutil

import pandas as pd
import numpy as np
import igraph as ig
from scipy.stats import rankdata, norm
from pyvis.network import Network
from IPython.display import HTML, display

import yaml

# Add repo root to path so we can import api
_repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from api import tetrad
tetrad.tetrad_jvm_start("resources/tetrad-current.jar")
from api import data, score, bootstrapping, algorithm, graph, knowledge

py_output_dir = "pykumu_outputs"
os.makedirs(py_output_dir, exist_ok=True)


/var/folders/qg/19_8zd9x78b4fcgq8j_d8jmc0000gn/T/ipykernel_73536/4249960605.py:23: UserWarning: JVM is already running. tetrad.start() has no effect after the first call — restart the Python process to change the JAR.
  tetrad.tetrad_jvm_start("resources/tetrad-current.jar")


This notebook performs the necessary data transformations to the final table generated by [issue_social_smell_showcase.Rmd](https://github.com/sailuh/kaiaulu/blob/master/vignettes/issue_social_smell_showcase.Rmd) Notebook in order to perform Causal Analysis using Pykumu and Tetrad via JPype.


# 3. Config

Use the configuration file to specify the necessary data inputs and causal model parameters for a given project. 

In [155]:
# dt = pd.read_csv("~/causal_tse/causal_modelling/1_openssl_social_smells_timeline.csv")

with open("../conf/openssl.yml", 'r') as file:
    conf = yaml.safe_load(file)

data_filepath = conf['project']["data_filepath"]
data_filepath


'resources/1_openssl_social_smells_timeline.csv'

The following is a sample of the data that will be used for causal analysis:

In [156]:
#dt = pd.read_csv("resources/null_variable_dt.csv")
dt = pd.read_csv(data_filepath)
dt.head()

,cve_id,commit_interval,start_day,end_day,org_silo,missing_links,radio_silence,primma_donna,st_congruence,communicability,code_only_devs,code_files,ml_only_devs,ml_threads,n_commits,sum_churn
0,CVE-2006-4339,d02b48c63a58ea4367a0e905979f140b7d090f86-abd4c...,1998-12-21T10:52:47Z,1999-03-21T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0
1,CVE-2006-4339,e778802f53c8d47e96a6e4cbc776eb6e1d4c461a-84c15...,1999-03-21T10:52:47Z,1999-06-19T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0
2,CVE-2006-4339,ce8b25741380eb08ca25ad06c2e83370067734ce-c1cd8...,1999-06-19T10:52:47Z,1999-09-17T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0
3,CVE-2006-4339,1c80019a2c8f59410552197723829fd72ab45a5e-1c800...,1999-09-17T10:52:47Z,1999-12-16T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0
4,CVE-2006-4339,dd9d233e2aa493fa1398b527afbf6aa5cdb23f23-59fc2...,1999-12-16T10:52:47Z,2000-03-15T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0


## 4. Feature Engineering

### 4.1 Formatting Data Types

In order to be loaded in Tetrad, some variables must be transformed from String to Integer due to data type limitations. 

#### 4.2 CVE Data Type

We concatenate the last two digits of the year with the last four digits of the cve_id and convert into an integer. (E.g. 2006 and CVE ID XXX4339 becomes 06339).


In [157]:
if 'cve_id' in dt.columns:
    last_two_digits_year = dt['cve_id'].str[6:8]
    last_four_digits_cve = dt['cve_id'].str[9:14]
    dt['cve_id'] = (last_two_digits_year + last_four_digits_cve).astype(int)
else:
    print("Skipped cve_id transformation (column missing or already numeric).")

In [158]:
dt.head()

,cve_id,commit_interval,start_day,end_day,org_silo,missing_links,radio_silence,primma_donna,st_congruence,communicability,code_only_devs,code_files,ml_only_devs,ml_threads,n_commits,sum_churn
0,64339,d02b48c63a58ea4367a0e905979f140b7d090f86-abd4c...,1998-12-21T10:52:47Z,1999-03-21T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0
1,64339,e778802f53c8d47e96a6e4cbc776eb6e1d4c461a-84c15...,1999-03-21T10:52:47Z,1999-06-19T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0
2,64339,ce8b25741380eb08ca25ad06c2e83370067734ce-c1cd8...,1999-06-19T10:52:47Z,1999-09-17T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0
3,64339,1c80019a2c8f59410552197723829fd72ab45a5e-1c800...,1999-09-17T10:52:47Z,1999-12-16T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0
4,64339,dd9d233e2aa493fa1398b527afbf6aa5cdb23f23-59fc2...,1999-12-16T10:52:47Z,2000-03-15T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0


Second, commit interval is transformed into `activity_0` and `activity_2` if the commit hash is missing or available respectively:

In [159]:
if 'commit_interval' in dt.columns:
    # R's data.table::fread reads empty CSV cells as the literal empty string,
    # so `commit_interval == ""` is TRUE there. Python's pd.read_csv reads them
    # as NaN, so the naive == "" check returns False. Treat both as missing:
    missing = dt['commit_interval'].isna() | (dt['commit_interval'] == "")
    dt['activity_0'] = missing.astype(int)
    dt['activity_2'] = (~missing).astype(int)
else:
    print("Skipped activity transformation (commit_interval column missing).")

# Match R: write feature_engineering.csv after activity transform but BEFORE
# any column renaming, so the file preserves the original column names.
dt.to_csv(os.path.join(py_output_dir, "feature_engineering.csv"), index=False)


In [160]:
dt.head()

,cve_id,commit_interval,start_day,end_day,org_silo,missing_links,radio_silence,primma_donna,st_congruence,communicability,code_only_devs,code_files,ml_only_devs,ml_threads,n_commits,sum_churn,activity_0,activity_2
0,64339,d02b48c63a58ea4367a0e905979f140b7d090f86-abd4c...,1998-12-21T10:52:47Z,1999-03-21T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0,0,1
1,64339,e778802f53c8d47e96a6e4cbc776eb6e1d4c461a-84c15...,1999-03-21T10:52:47Z,1999-06-19T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0,0,1
2,64339,ce8b25741380eb08ca25ad06c2e83370067734ce-c1cd8...,1999-06-19T10:52:47Z,1999-09-17T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0,0,1
3,64339,1c80019a2c8f59410552197723829fd72ab45a5e-1c800...,1999-09-17T10:52:47Z,1999-12-16T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0,0,1
4,64339,dd9d233e2aa493fa1398b527afbf6aa5cdb23f23-59fc2...,1999-12-16T10:52:47Z,2000-03-15T10:52:47Z,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0,0,1


## Feature Renaming

A number of feature names are also shortened, so their visual representation do not take too much screen space:


In [161]:
rename_map = {
    "start_day": "start",
    "missing_links": "mis_link",
    "radio_silence": "silence",
    "code_only_devs": "code_dev",
    "code_files": "file",
    "ml_only_devs": "mail_dev",
    "ml_threads": "thread",
    "n_commits": "commit",
    "sum_churn": "churn"
}

cols_to_rename = {k: v for k, v in rename_map.items() if k in dt.columns}
if cols_to_rename:
    dt = dt.rename(columns=cols_to_rename)
    expected_cols = ["cve_id", "activity_0", "activity_2", "start", "org_silo",
                     "mis_link", "silence", "code_dev", "file", "mail_dev",
                     "thread", "commit", "churn"]
    available_cols = [c for c in expected_cols if c in dt.columns]
    dt = dt[available_cols]
else:
    print("Skipped feature renaming (columns already renamed or missing).")

dt.to_csv(os.path.join(py_output_dir, "feature_renaming.csv"), index=False)


In [162]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,1998-12-21T10:52:47Z,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0
1,64339,0,1,1999-03-21T10:52:47Z,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0
2,64339,0,1,1999-06-19T10:52:47Z,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0
3,64339,0,1,1999-09-17T10:52:47Z,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0
4,64339,0,1,1999-12-16T10:52:47Z,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0


### Missing Data Handling

We decided to remove rows from the dataset for which the mailing list data source is missing (i.e. 2000-2001).

In [163]:
if 'start' in dt.columns:
    dt['start'] = pd.to_datetime(dt['start'])
    # R converts POSIXct -> as.numeric inside the Missing Data Handling block,
    # before writing missing_data_handling.csv. Do the conversion here so the
    # downstream CSV writes match R's column type.
    if dt['start'].dt.tz is not None:
        dt['start'] = dt['start'].dt.tz_convert('UTC').dt.tz_localize(None)
    dt['start'] = dt['start'].astype('datetime64[s]').astype('int64')
    print(f"Parsed and converted start to Unix seconds (rows kept: {len(dt)}).")
else:
    print("Skipped start parsing (column missing).")


Parsed and converted start to Unix seconds (rows kept: 6697).


In [164]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0
1,64339,0,1,922013567,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0
2,64339,0,1,929789567,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0
3,64339,0,1,937565567,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0
4,64339,0,1,945341567,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0


In [165]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,NaN,NaN,NaN,3.0,4.0,NaN,NaN,13.0,2935.0
1,64339,0,1,922013567,NaN,NaN,NaN,4.0,4.0,NaN,NaN,9.0,302.0
2,64339,0,1,929789567,NaN,NaN,NaN,2.0,3.0,NaN,NaN,5.0,60.0
3,64339,0,1,937565567,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,171.0
4,64339,0,1,945341567,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,21.0


With respect to data missing due to inactivity during a given time period, any measures of features (counts) related to commits should all be 0.

In [166]:
if dt.isna().any().any():
    dt = dt.fillna(0)
    print("Applied fillna(0) â€” missing values found and filled.")
else:
    print("Skipped fillna (no missing values found).")

dt.to_csv(os.path.join(py_output_dir, "missing_data_handling.csv"), index=False)


Applied fillna(0) â€” missing values found and filled.


In [167]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,13.0,2935.0
1,64339,0,1,922013567,0.0,0.0,0.0,4.0,4.0,0.0,0.0,9.0,302.0
2,64339,0,1,929789567,0.0,0.0,0.0,2.0,3.0,0.0,0.0,5.0,60.0
3,64339,0,1,937565567,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,171.0
4,64339,0,1,945341567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,3.0,21.0


#### Convert "start" to Unix Timestamp

To use start in causal analysis, we convert it to a unix timestamp. 

In [168]:
# start is already Unix-timestamp from the Missing Data Handling cell.
# Kept here as a comment marker; conversion happens earlier.
print("start already in Unix seconds.")


start already in Unix seconds.


In [169]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,13.0,2935.0
1,64339,0,1,922013567,0.0,0.0,0.0,4.0,4.0,0.0,0.0,9.0,302.0
2,64339,0,1,929789567,0.0,0.0,0.0,2.0,3.0,0.0,0.0,5.0,60.0
3,64339,0,1,937565567,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,171.0
4,64339,0,1,945341567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,3.0,21.0


### 1-Time Lag Features

In [170]:
lag_cols = ["org_silo", "mis_link", "silence", "code_dev", "file",
            "mail_dev", "thread", "commit", "churn"]

def add_time_lag(cve_table):
    if len(cve_table) < 2:
        for col in lag_cols:
            cve_table[col + "2"] = np.nan
        return cve_table
    else:
        current = cve_table.iloc[:-1].reset_index(drop=True)
        future = cve_table[lag_cols].iloc[1:].reset_index(drop=True)
        future.columns = [col + "2" for col in lag_cols]
        return pd.concat([current, future], axis=1)

In [171]:
if 'cve_id' in dt.columns and not any(c.endswith('2') for c in dt.columns if c not in ['activity_2']):
    # Iterate groups explicitly so the cve_id column is preserved on each
    # sub-DataFrame passed to add_time_lag. Newer pandas excludes the grouping
    # column from what groupby.apply() passes through, which silently drops it.
    sorted_dt = dt.sort_values(["cve_id", "start"])
    pieces = [add_time_lag(grp.copy()) for _, grp in sorted_dt.groupby("cve_id", sort=False)]
    lag_dt = pd.concat(pieces, ignore_index=True)
    print("Applied time lag features.")
else:
    lag_dt = dt.copy()
    print("Skipped time lag (lag columns already present or cve_id missing).")

lag_dt.to_csv(os.path.join(py_output_dir, "time_lag_features.csv"), index=False)


Applied time lag features.


In [172]:
lag_dt.columns

Index(['cve_id', 'activity_0', 'activity_2', 'start', 'org_silo', 'mis_link',
       'silence', 'code_dev', 'file', 'mail_dev', 'thread', 'commit', 'churn',
       'org_silo2', 'mis_link2', 'silence2', 'code_dev2', 'file2', 'mail_dev2',
       'thread2', 'commit2', 'churn2'],
      dtype='str')

### Remove Short CVEs 

We deleted CVEs (their associated rows) with 7 or fewer time periods.

In [173]:
if 'cve_id' in lag_dt.columns:
    short_cves = (lag_dt.groupby("cve_id").size()
                  .reset_index(name="n_rows")
                  .sort_values("n_rows")
                  .query("n_rows <= 7"))
    print("Identified short CVEs:")
    print(short_cves)
else:
    short_cves = pd.DataFrame(columns=["cve_id", "n_rows"])
    print("Skipped short CVE detection (cve_id column missing).")

short_cves.to_csv(os.path.join(py_output_dir, "short_cves.csv"), index=False)


Identified short CVEs:
     cve_id  n_rows
105  167054       2
101  166307       3
102  166309       3
99   166305       4
11   101633       6
10   100742       6
115  191543       7


In [174]:
if 'cve_id' in lag_dt.columns and len(short_cves) > 0:
    short_cve_ids = short_cves['cve_id'].values
    lag_dt = lag_dt[~lag_dt['cve_id'].isin(short_cve_ids)]
    print(f"Removed {len(short_cve_ids)} short CVEs.")
else:
    print("Skipped short CVE removal (no short CVEs or cve_id missing).")

Removed 7 short CVEs.


## Addressing Determinism and High Intercorrelation Among Features

In [175]:
cor_cols = ["org_silo", "mis_link", "silence", "code_dev", "file",
            "mail_dev", "thread", "commit", "churn",
            "org_silo2", "mis_link2", "silence2", "code_dev2", "file2",
            "mail_dev2", "thread2", "commit2", "churn2"]

available_cor_cols = [c for c in cor_cols if c in lag_dt.columns]
if available_cor_cols:
    cor_table = lag_dt[available_cor_cols]
    cor_table.corr()
else:
    print("Skipped correlation analysis (columns not present).")

cor_matrix = lag_dt[available_cor_cols].corr()
cor_matrix.insert(0, "rn", cor_matrix.index)
cor_matrix.to_csv(os.path.join(py_output_dir, "correlation_matrix.csv"), index=False)


Due to high correlation, we perform 6 feature deletions (activity_0, activity_2, org_silo, org_silo2):

In [176]:
keep_cols = ["cve_id", "start", "mis_link", "silence", "code_dev", "file",
             "mail_dev", "thread", "commit", "churn",
             "mis_link2", "silence2", "code_dev2", "file2",
             "mail_dev2", "thread2", "commit2", "churn2"]

available_keep = [c for c in keep_cols if c in lag_dt.columns]
if 'org_silo' in lag_dt.columns or 'activity_0' in lag_dt.columns:
    lag_dt = lag_dt[available_keep]
    print("Applied feature deletion (removed correlated columns).")
else:
    print("Skipped feature deletion (columns already removed).")

Applied feature deletion (removed correlated columns).


In [177]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,13.0,2935.0
1,64339,0,1,922013567,0.0,0.0,0.0,4.0,4.0,0.0,0.0,9.0,302.0
2,64339,0,1,929789567,0.0,0.0,0.0,2.0,3.0,0.0,0.0,5.0,60.0
3,64339,0,1,937565567,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,171.0
4,64339,0,1,945341567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,3.0,21.0


## Non-Parametric Transformation

We then apply the non-paranormal distribution transformation to numerical variables, to reduce the risk of violating the normal distribution when drawing causal conclusions.

In [178]:
def npn(data):
    """Non-paranormal (nonparanormal) transformation matching R's huge::npn default.

    Replicates the installed huge::npn(npn.func="shrinkage") behavior:

        x = qnorm(apply(x, 2, rank) / (n + 1))
        # normalize.columns: divide each column by its own SD (floor at 1
        # if SD is zero or non-finite)
        x = sweep(x, 2, apply(x, 2, sd), "/")

    Note: the CRAN-published older source divides all columns by sd(x[, 1])
    instead. The installed package's per-column normalization is what the R
    notebook is actually running on this dataset.
    """
    n = len(data)
    result = data.copy()
    for col in data.columns:
        ranks = rankdata(data[col]) / (n + 1)
        result[col] = norm.ppf(ranks)
    for col in result.columns:
        s = result[col].std(ddof=1)
        if not np.isfinite(s) or s == 0:
            s = 1.0
        result[col] = result[col] / s
    return result

if 'cve_id' in lag_dt.columns:
    lag_dt = pd.concat([lag_dt[["cve_id", "start"]],
                        npn(lag_dt.iloc[:, 2:])], axis=1)
    print("Applied non-paranormal transformation.")
    lag_dt.head()
else:
    print("Skipped non-paranormal transformation (data already transformed).")
    lag_dt.head()

lag_dt.to_csv(os.path.join(py_output_dir, "npn_transformed.csv"), index=False)


Applied non-paranormal transformation.


### Binarized CVE Indicators

To represent the CVE Ids, we utilize indicator features. For every CVE ID, a new column is added to the table which can take values 0 or 1. The value is 1 if the row is associated to that CVE ID, or 0 otherwise.

In [179]:
if 'cve_id' in lag_dt.columns:
    binarize_cve_id = pd.DataFrame({
        "id": range(len(lag_dt)),
        "cve_id": "b_" + lag_dt["cve_id"].astype(str),
        "binary_value": 1
    })
    binarize_cve_id = binarize_cve_id.pivot(index="id", columns="cve_id",
                                            values="binary_value").fillna(0).astype(int)
    binarize_cve_id = binarize_cve_id.reset_index()
    pd.concat([lag_dt[["cve_id"]].reset_index(drop=True), binarize_cve_id], axis=1).head()
    print("Applied CVE binarization.")
else:
    binarize_cve_id = None
    print("Skipped CVE binarization (cve_id column missing â€” data already binarized).")

if binarize_cve_id is not None:
    pd.concat([lag_dt[["cve_id"]].reset_index(drop=True), binarize_cve_id], axis=1).to_csv(os.path.join(py_output_dir, "binarized_cve_indicators.csv"), index=False)


Applied CVE binarization.


We can then remove the `cve_id` column, as the binary features represent the same information, and add the remaining columns to the analysis table:

In [180]:
if binarize_cve_id is not None:
    lag_dt = lag_dt[["start", "mis_link", "silence", "code_dev", "file",
                    "mail_dev", "thread", "commit", "churn",
                    "mis_link2", "silence2", "code_dev2", "file2",
                    "mail_dev2", "thread2", "commit2", "churn2"]].reset_index(drop=True)
    binarized_lag_dt = pd.concat([lag_dt, binarize_cve_id.drop(columns="id")], axis=1)
    print("Applied cve_id removal and binary column concatenation.")
else:
    # Data already has binarized columns â€” extract non-nv columns as binarized_lag_dt
    nv_cols = [c for c in lag_dt.columns if c.startswith("nv-")]
    binarized_lag_dt = lag_dt.drop(columns=nv_cols, errors='ignore')
    print("Skipped â€” using existing binarized columns from loaded data.")

binarized_lag_dt.to_csv(os.path.join(py_output_dir, "binarized_lag_dt.csv"), index=False)


Applied cve_id removal and binary column concatenation.


In [181]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,13.0,2935.0
1,64339,0,1,922013567,0.0,0.0,0.0,4.0,4.0,0.0,0.0,9.0,302.0
2,64339,0,1,929789567,0.0,0.0,0.0,2.0,3.0,0.0,0.0,5.0,60.0
3,64339,0,1,937565567,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,171.0
4,64339,0,1,945341567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,3.0,21.0


### Add Null Features

An example of the randomization only showing the silence and nv-silence is shown below. In practice, for every column in `lag_dt` up to this point, we generated a replica column prefixed by `nv-`, including the binary features (which are then prefixed as `nv-b_`), but the replica columns have their values shuffled across the rows, hence the null (random) naming to them.


In [182]:
if not any(c.startswith("nv-") for c in lag_dt.columns):
    nv_lag_dt = binarized_lag_dt.copy()
    nv_lag_dt.columns = "nv-" + binarized_lag_dt.columns
    # Seed each column shuffle deterministically. R sets set.seed(1) at the top
    # of the notebook so its `apply(nv_lag_dt, 2, sample)` is reproducible.
    # Note: R's and NumPy's Mersenne-Twister output bytes diverge, so the
    # specific shuffled values will NOT byte-match R even with the same seed;
    # but Python is at least reproducible run-to-run and statistically
    # equivalent (each column is a uniform random permutation).
    rng = np.random.default_rng(1)
    nv_lag_dt = nv_lag_dt.apply(lambda col: pd.Series(rng.permutation(col.values)))
    nv_lag_dt = pd.concat([binarized_lag_dt, nv_lag_dt], axis=1)
    print("Applied null variable features.")
    nv_lag_dt[["silence", "nv-silence"]].head()
else:
    nv_lag_dt = lag_dt.copy()
    print("Skipped null feature creation (nv- columns already present).")
    if "silence" in nv_lag_dt.columns and "nv-silence" in nv_lag_dt.columns:
        nv_lag_dt[["silence", "nv-silence"]].head()

Applied null variable features.


In [183]:
dt.head()

,cve_id,activity_0,activity_2,start,org_silo,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn
0,64339,0,1,914237567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,13.0,2935.0
1,64339,0,1,922013567,0.0,0.0,0.0,4.0,4.0,0.0,0.0,9.0,302.0
2,64339,0,1,929789567,0.0,0.0,0.0,2.0,3.0,0.0,0.0,5.0,60.0
3,64339,0,1,937565567,0.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0,171.0
4,64339,0,1,945341567,0.0,0.0,0.0,3.0,4.0,0.0,0.0,3.0,21.0


### Keep only 5 null indicator features

Introducing a null feature for all variables and features leads to too many features being introduced for causal search, causing heap memory errors in Tetrad. We preserve only a few of the nv binary indicator variables, as they lead to variable explosion and their pattern is easy to randomize. Position 138 includes all variables as null variables, plus five binary indicators as null variables. We consider this loss of null binary indicator features reasonable, as the randomization of a few blocks of values 1 or 0 will generally be equivalent. This in turn, allow us to perform more causal search runs, which we deem a fair trade-off. 

In [184]:
if nv_lag_dt.shape[1] > 138:
    nv_lag_dt = nv_lag_dt.iloc[:, :138]
    print(f"Trimmed to 138 columns.")
else:
    print(f"Skipped column trimming (already {nv_lag_dt.shape[1]} columns).")

# Convert all columns to float so Tetrad treats them as continuous (required for SEM BIC)
nv_lag_dt = nv_lag_dt.astype(float)
binarized_lag_dt = binarized_lag_dt.astype(float)
print(f"Converted nv_lag_dt and binarized_lag_dt to float.")

nv_lag_dt.to_csv(os.path.join(py_output_dir, "nv_lag_dt_final.csv"), index=False)


Trimmed to 138 columns.
Converted nv_lag_dt and binarized_lag_dt to float.


In [185]:
print(nv_lag_dt)

             start  mis_link   silence  code_dev      file  mail_dev  \
0     9.763026e+08 -0.440884 -0.840522  0.560444  0.153841 -1.434590   
1     9.840786e+08 -0.440884 -0.840522  0.560444  0.153841 -1.434590   
2     9.918546e+08 -0.440884 -0.840522 -0.084979  0.153841 -1.434590   
3     9.996306e+08 -0.440884 -0.840522 -1.085997 -1.109931 -1.434590   
4     1.007407e+09 -0.440884 -0.078923 -0.084979  0.153841 -0.981523   
...            ...       ...       ...       ...       ...       ...   
6540  1.466334e+09  3.209247 -0.078923  2.234493  0.153841 -0.888254   
6541  1.474110e+09 -0.440884 -0.840522  1.533115  0.153841 -1.434590   
6542  1.481886e+09 -0.440884 -0.840522  2.530418  0.153841 -1.434590   
6543  1.489662e+09  2.757531 -0.085622  1.920547  0.153841 -0.989958   
6544  1.497438e+09 -0.440884 -0.840522  2.234493  0.153841 -1.434590   

        thread    commit     churn  mis_link2  ...  b_81672  b_85077  b_93245  \
0    -1.434412  0.809623  1.348384  -0.452010  ...    

## FGES Null Variable Search

Since we are using JPype and TetradSearch directly (instead of causal-cmd), we do not need to save the dataset to CSV. Instead, we pass the pandas DataFrame directly to Tetrad.

We then set the output file configuration. 


In [186]:
# Output path configuration (for saving results later if needed)
output_folder_path = os.path.join(os.getcwd(), "null_search")

Finally, we perform causal search over our null dataset. In this Notebook we include both FGES and BOSS. To execute one or the other, modify `eval` to TRUE on either code block. This may take some time to execute. After the data is generated, the code block evaluation can be again set to FALSE, as the remaining analysis can be performed using the output file of either algorithms. 

The output `.json` file of Causal Search can also be loaded directly on Tetrad GUI for inspection.




In [187]:
# Set eval_fges = True to run, False to skip
eval_fges = True

if eval_fges:
    filename = "fges_bootstrap_null_search_500_runs_nv_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.transform_pandasdf_to_tetrad_boxdataset(nv_lag_dt)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.bootstrapping(state["params"], number_resampling=10,
                                    percent_resample_size=90, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.algorithm_fges(state["data"], state["params"], sem_bic, state["knowledge"],
                                symmetric_first_step=True, max_degree=25,
                                faithfulness_assumed=True, parallelized=False)

    # Save graph as JSON
    fges_null_graph_json = graph.transform_graph_java_to_graph_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(fges_null_graph_json))
    print(f"FGES null search graph saved to: {filepath}")

if eval_fges:
    shutil.copy2(filepath, os.path.join(py_output_dir, "null_search_graph.json"))


Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
FGES null search graph saved to: /Users/phuonghuupham/pykumu/notebook/null_search/fges_bootstrap_null_search_500_runs_nv_binary_indicators_graph.json


## BOSS Null Variable Search

This is the BOSS Search. We found BOSS scaled better, allowing us to increase the number of bootstraps up to 1000. Observe also the knowledge box is not specified at this point: We do not impose any restrictions when observing the formation of edges at random. Our final conclusions are derived from BOSS. Note also that we bootstrap on BOSS with 100% of the dataset, while in FGES we do so with 90% of the dataset. We use 100% in BOSS, because the algorithm already has random initialization. 


In [188]:
# Set eval_boss = True to run, False to skip
eval_boss = False

if eval_boss:
    filename = "boss_bootstrap_null_search_1000_runs_nv_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.transform_pandasdf_to_tetrad_boxdataset(nv_lag_dt)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.bootstrapping(state["params"], number_resampling=10,
                                    percent_resample_size=100, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.algorithm_boss(state["data"], state["params"], sem_bic, state["knowledge"],
                                num_starts=1, use_bes=False, time_lag=0,
                                use_data_order=True)

    # Save graph as JSON
    boss_null_graph_json = graph.transform_graph_java_to_graph_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(boss_null_graph_json))
    print(f"BOSS null search graph saved to: {filepath}")

if eval_boss:
    shutil.copy2(filepath, os.path.join(py_output_dir, "null_search_graph.json"))


## Deriving the 1 PNEF Threshold

In our causal search above, we introduced null features over multiple bootstrap runs to observe how often our causal search form random edges (i.e. between our features and null features). We will use this information to derive a threshold, 1PNEF, we can use in our final causal search.


### Graph Examination

We now have our causal bootstrap graph as a .json file, which is output by Tetrad. Let's parse it into a tabular format to provide further intuition on how the 1 PNEF threshold is being determined. 

The nodes contain all our variables and null features In the off_chance a feature does not have any edge to it, this table allow us to still show it on the graph, as it would not appear on the "edge list" table.


In [189]:
parsed_graph = graph.parse_graph_json(filepath)
parsed_graph["nodes"].head()

parsed_graph["nodes"].to_csv(os.path.join(py_output_dir, "null_search_nodes.csv"), index=False)


Next is the edgeset table output by Tetrad. This table contains all the edges. Because we are performing multiple executions, each with a sample of the full dataset (as we are using a "bootstrap" approach), the probabilities represented here are the "ensemble" of all edges formed on each execution. In this Notebook, the preserved ensemble was used.


In [190]:
parsed_graph["edgeset"].head()

parsed_graph["edgeset"].to_csv(os.path.join(py_output_dir, "null_search_edgeset.csv"), index=False)


Lastly, we can examine the counts of each type of edge formed on each subgraph via the edge_type_probabilities table. Since the edgeset table probability already sums the probabilities from this table for every node pair, this information is presented here only for qualitative inspection, but it is not currently used in the subsequent steps.


In [191]:
parsed_graph["edge_type_probabilities"].head()

parsed_graph["edge_type_probabilities"].to_csv(os.path.join(py_output_dir, "null_search_edge_type_probabilities.csv"), index=False)


### Deriving 1 PNEF

As noted, our interest is to derive a threshold for the final causal search, using the information of this bootstrapped null feature causal search between the actual variables, and the random features. By this randome dge definition, our first step is to subset the `edgeset`table  to contain only the edge pairs that include null variables. A sample is shown below of the table where at least one of the two nodes is nv:


In [192]:
is_node1_nv = parsed_graph["edgeset"]["node1_name"].str.contains("nv-", regex=False)
is_node2_nv = parsed_graph["edgeset"]["node2_name"].str.contains("nv-", regex=False)
nv_edges = parsed_graph["edgeset"][is_node1_nv | is_node2_nv].copy()
nv_edges.head()

nv_edges.to_csv(os.path.join(py_output_dir, "nv_edges.csv"), index=False)


Next, we can derive a no_edge probability by subtracting 1 from the `probability` value. 


In [193]:
nv_edges["no_edge"] = 1 - nv_edges["probability"]
nv_edges.head()

,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge


Our goal then is to identify the first percentile value of the no edge probability, i.e. the 1st percentile NoEdge Frequency value (1PNEF):


In [194]:
if len(nv_edges) > 0:
    pnef_1 = float(nv_edges["no_edge"].quantile(0.01))
else:
    # No nv-* edges formed in the null search, so the noise distribution is
    # empty and the 1st-percentile is undefined. Fall back to the strictest
    # filter that still produces a non-empty graph: keep only edges that
    # appeared in 100% of bootstrap runs (no_edge == 0.0).
    pnef_1 = 0.0
    print("WARNING: no nv-* edges found; defaulting pnef_1 = 0.0 (keeping only edges with bootstrap probability = 1.0).")
pnef_1

with open(os.path.join(py_output_dir, "pnef_1.txt"), "w") as f:
    f.write(str(pnef_1) + "\n")


What this threshold tell us is that, if executed 1000 runs, then the first percentile of all random edges formed had approximately 65% no formation of causal link. Another way to state this is that given entirely random variables, causal links were formed between them up to 35% of the time. In our final search, we then only keep causal links that, over 1000 runs, formed **more** than 35% of the time, under the assumption any causal link established less than that may be due to random chance. 

With the threshold defined, we can now proceed to the final causal search, which does not include null features. In this non null feature causal search, we also specify domain knowledge. 

## Non-Null Causal Search

### Domain Knowledge Causal Search without Null Variables

Domain knowledge is used to prohibit causal links to form among features. Here, we only defined temporal causal link restrictions. I.e. it does not make sense for features at 1-time-lag (future) to cause features on the present time.


In [195]:
#knowledge_file_path = os.path.expanduser("~/Downloads/knowledge_2.txt")
knowledge_file_path = ("resources/knowledge_box.txt")  

### Causal Search

As before, we specify the graph output file. 

In [196]:
output_folder_path = os.path.join(os.getcwd(), "domain_binarized_search")

We also have the choice of using FGES or BOSS here. In our final analysis, we used BOSS.

FGES Causal Search:

In [197]:
# Set eval_fges_domain = True to run, False to skip
eval_fges_domain = False

if eval_fges_domain:
    filename = "fges_bootstrap_binarized_search_500_runs_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.transform_pandasdf_to_tetrad_boxdataset(binarized_lag_dt)
    state["knowledge"] = knowledge.parse_knowledge_txt(knowledge_file_path)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.bootstrapping(state["params"], number_resampling=10,
                                    percent_resample_size=90, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.algorithm_fges(state["data"], state["params"], sem_bic, state["knowledge"],
                                symmetric_first_step=True, max_degree=1000,
                                faithfulness_assumed=True, parallelized=False)

    # Save graph as JSON
    fges_domain_graph_json = graph.transform_graph_java_to_graph_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(fges_domain_graph_json))
    print(f"FGES domain search graph saved to: {filepath}")

if eval_fges_domain:
    shutil.copy2(filepath, os.path.join(py_output_dir, "domain_search_graph.json"))


BOSS Causal Search:


In [198]:
# Set eval_boss_domain = True to run, False to skip
eval_boss_domain = True

if eval_boss_domain:
    filename = "boss_bootstrap_binarized_search_1000_runs_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.transform_pandasdf_to_tetrad_boxdataset(binarized_lag_dt)
    state["knowledge"] = knowledge.parse_knowledge_txt(knowledge_file_path)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.bootstrapping(state["params"], number_resampling=10,
                                    percent_resample_size=100, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.algorithm_boss(state["data"], state["params"], sem_bic, state["knowledge"],
                                num_starts=1, use_bes=False, time_lag=0,
                                use_data_order=True, num_threads=5)

    # Save graph as JSON
    boss_domain_graph_json = graph.transform_graph_java_to_graph_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(boss_domain_graph_json))
    print(f"BOSS domain search graph saved to: {filepath}")

if eval_boss_domain:
    shutil.copy2(filepath, os.path.join(py_output_dir, "domain_search_graph.json"))



Loading knowledge.
Adding to tier 1 start
Adding to tier 1 mis_link
Adding to tier 1 silence
Adding to tier 1 code_dev
Adding to tier 1 file
Adding to tier 1 mail_dev
Adding to tier 1 thread
Adding to tier 1 commit
Adding to tier 1 churn
Adding to tier 2 mis_link2
Adding to tier 2 silence2
Adding to tier 2 code_dev2
Adding to tier 2 file2
Adding to tier 2 mail_dev2
Adding to tier 2 thread2
Adding to tier 2 commit2
Adding to tier 2 churn2
Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
BOSS domain search graph saved to: /Users/phuonghuupham/pykumu/notebook/domain_binarized_search/boss_bootstrap_binarized_search_1000_runs_binary_indicators_graph.json


## Applying 1PNEF Threshold

We load the final causal search, and then apply the 1PNEF threshold derived from the prior causal search here. A sample of the causal graph nodes and edges is shown below:


In [199]:
parsed_graph = graph.parse_graph_json(filepath)
print("Nodes:")
print(parsed_graph["nodes"].head())
print("\nEdgeset:")
print(parsed_graph["edgeset"].head())
print("\nEdge Type Probabilities:")
print(parsed_graph["edge_type_probabilities"].head())

parsed_graph["nodes"].to_csv(os.path.join(py_output_dir, "final_search_nodes.csv"), index=False)
parsed_graph["edgeset"].to_csv(os.path.join(py_output_dir, "final_search_edgeset.csv"), index=False)
parsed_graph["edge_type_probabilities"].to_csv(os.path.join(py_output_dir, "final_search_edge_type_probabilities.csv"), index=False)


Nodes:
  node_name
0  b_100433
1  b_100740
2  b_102939
3  b_103864
4  b_104180

Edgeset:
  node1_name node2_name endpoint1 endpoint2   bold  highlighted properties  \
0    b_62940    commit2      TAIL     ARROW  False        False      dd;pl   
1   b_140224       file      TAIL     ARROW  False        False      dd;pl   
2    b_63738      file2      TAIL     ARROW  False        False      dd;pl   
3      start     thread      TAIL     ARROW  False        False      pd;nl   
4      start   b_160705      TAIL     ARROW  False        False      pd;nl   

   probability  
0     0.818182  
1     1.000000  
2     1.000000  
3     0.909091  
4     0.909091  

Edge Type Probabilities:
  node1_name node2_name edge_type properties  probability
0    b_62940    commit2        ta      dd;pl     0.818182
1    b_62940    commit2       nil        NaN     0.181818
2   b_140224       file        ta      dd;pl     1.000000
3    b_63738      file2        ta      dd;pl     1.000000
4      start     thread 

### Applying 1PNEF Threshold

Edges which may have been formed at random are filtered here:

In [200]:
edges = parsed_graph["edgeset"].copy()
edges["no_edge"] = 1 - edges["probability"]
edges_1pnef = edges[edges["no_edge"] <= pnef_1].copy()
edges_1pnef

edges_1pnef.to_csv(os.path.join(py_output_dir, "edges_1pnef.csv"), index=False)


## Results 

With the final causal graph trimmed, we can now inspect it to draw conclusions from it. Causal graphs may form cycles, and also have undirected edges. We define a function to check for cycles here and use it below for inspection.


In [201]:
# colorBlindness::Blue2DarkRed12Steps (verified from R/palette.R in jianhong/colorBlindness)
Blue2DarkRed12Steps = [
    "#290AD8", "#264DFF", "#3FA0FF", "#72D9FF", "#AAF7FF", "#E0FFFF",
    "#FFFFBF", "#FFE099", "#FFAD72", "#F76D5E", "#D82632", "#A50021",
]


# Python equivalent of R's visIgraph(g, randomSeed=1) %>% visOptions %>% visInteraction.
# Layout via python-igraph FR (same C library as R), render via pyvis (vis.js wrapper).
def vis_igraph(nodes, edges, output_path, seed=1):
    names = list(nodes['node'])
    idx = {n: i for i, n in enumerate(names)}
    pairs = [(idx[f], idx[t]) for f, t in zip(edges['from'], edges['to']) if f in idx and t in idx]

    g = ig.Graph(n=len(names), edges=pairs, directed=True)
    random.seed(seed); ig.set_random_number_generator(random)
    coords = g.layout_fruchterman_reingold(niter=500).coords
    pos = {names[i]: (x * 200, -y * 200) for i, (x, y) in enumerate(coords)}

    net = Network(height="700px", width="100%", directed=True, notebook=True, cdn_resources='in_line')
    for _, r in nodes.iterrows():
        x, y = pos[r['node']]
        net.add_node(r['node'], label=r['node'], color=r['color'], title=r['node'],
                     size=40, x=x, y=y, physics=False)
    for f, t, color, weight in zip(edges['from'], edges['to'], edges['color'], edges['weight']):
        net.add_edge(f, t, color=color, title=f"p={weight:.3f}", arrows='to')
    net.set_options('{"physics": {"enabled": false}, '
                    '"interaction": {"navigationButtons": true, "keyboard": true, "hover": true}, '
                    '"edges": {"smooth": false, "arrowStrikethrough": false}}')

    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    net.save_graph(output_path)

    # Embed inline so the graph travels with the notebook output
    # (works in interactive Jupyter, in the make docs-notebook HTML export,
    # and when the .ipynb is shared standalone)
    with open(output_path) as f:
        encoded = base64.b64encode(f.read().encode()).decode()
    display(HTML(
        f'<iframe src="data:text/html;base64,{encoded}" '
        f'width="100%" height="700" frameborder="0"></iframe>'
    ))


### Full Causal Graph 1-PNEF Trimmed 

First, we can inspect the full causal graph.

In [202]:
nodes = parsed_graph['nodes'].rename(columns={'node_name': 'node'}).copy()

edges = edges_1pnef.copy()
edges['color'] = 'black'
edges.loc[(edges['endpoint1'] == 'TAIL') & (edges['endpoint2'] == 'TAIL'), 'color'] = 'red'
edges = edges.rename(columns={'node1_name': 'from', 'node2_name': 'to'})
edges['weight'] = edges['probability']
edges = edges[['from', 'to', 'color', 'weight']]

nodes_n = nodes.copy()
nodes_n['color'] = Blue2DarkRed12Steps[2]
nodes_n.loc[nodes_n['node'].isin(['silence',  'silence2']),  'color'] = Blue2DarkRed12Steps[4]
nodes_n.loc[nodes_n['node'].isin(['mis_link', 'mis_link2']), 'color'] = Blue2DarkRed12Steps[2]
nodes_n.loc[nodes_n['node'].isin(['code_dev', 'code_dev2']), 'color'] = Blue2DarkRed12Steps[6]
nodes_n.loc[nodes_n['node'].isin(['churn',    'churn2']),    'color'] = Blue2DarkRed12Steps[7]
nodes_n.loc[nodes_n['node'].isin(['commit',   'commit2']),   'color'] = Blue2DarkRed12Steps[8]
nodes_n.loc[nodes_n['node'].str.contains('b_', regex=False), 'color'] = Blue2DarkRed12Steps[0]

vis_igraph(nodes_n, edges, "causal_graph/causal_graph_full.html")


/Users/phuonghuupham/Library/Python/3.13/lib/python/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Sub-Graphs of Effort Variables and Parents


In [203]:
nodes_of_interest = ["silence", "silence2", "mis_link", "mis_link2",
                     "code_dev", "code_dev2", "churn", "churn2",
                     "commit", "commit2"]
edges_n = edges[edges['from'].isin(nodes_of_interest) & edges['to'].isin(nodes_of_interest)].copy()
nodes_n = nodes[nodes['node'].isin(set(edges_n['from']) | set(edges_n['to']))].copy()

nodes_n['color'] = Blue2DarkRed12Steps[2]
nodes_n.loc[nodes_n['node'].isin(['silence',  'silence2']),  'color'] = Blue2DarkRed12Steps[4]
nodes_n.loc[nodes_n['node'].isin(['mis_link', 'mis_link2']), 'color'] = Blue2DarkRed12Steps[2]
nodes_n.loc[nodes_n['node'].isin(['code_dev', 'code_dev2']), 'color'] = Blue2DarkRed12Steps[6]
nodes_n.loc[nodes_n['node'].isin(['churn',    'churn2']),    'color'] = Blue2DarkRed12Steps[7]
nodes_n.loc[nodes_n['node'].isin(['commit',   'commit2']),   'color'] = Blue2DarkRed12Steps[8]

vis_igraph(nodes_n, edges_n, "causal_graph/causal_graph_subgraph.html")


/Users/phuonghuupham/Library/Python/3.13/lib/python/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [204]:
names = sorted(set(edges['from']) | set(edges['to']))
idx = {n: i for i, n in enumerate(names)}
g = ig.Graph(n=len(names),
             edges=[(idx[f], idx[t]) for f, t in zip(edges['from'], edges['to'])],
             directed=True)
g.vs["name"] = names

graph.find_cycles(g)


[]